# DCCN Chits — Extracted Python Codes

Codes transcribed from the uploaded PDF. Each program is kept in a separate cell. Some snippets require `matplotlib` (`pip install matplotlib`).

## 1. Manchester and Differential Manchester Encoding

In [ ]:
import matplotlib.pyplot as plt

bits = input("Enter bits: ")

def manchester(bits):
    a = []
    for b in bits:
        a += [-1, 1] if b == '1' else [1, -1]
    return a

def diff_manchester(bits):
    a, cur = [], -1
    for b in bits:
        first = -cur if b == '0' else cur
        a += [first, -first]
        cur = -first
    return a

def plot(bits, y, title, ax):
    x = [i/2 for i in range(len(y)+1)]
    ax.step(x, y + [y[-1]], where="post")
    for i, b in enumerate(bits):
        ax.text(i+0.5, 1.5, b, ha='center')
    ax.axhline(0, color="gray", linestyle="--")
    ax.set_ylim(-2, 2)
    ax.set_yticks([-1, 0, 1])
    ax.set_xticks(range(len(bits)+1))
    ax.set_title(title)
    ax.grid(True, axis='x')

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
plot(bits, manchester(bits), "Manchester", ax[0])
plot(bits, diff_manchester(bits), "Differential Manchester", ax[1])
plt.tight_layout()
plt.show()

## 2. HDB3 and B8ZS Encoding

In [ ]:
import matplotlib.pyplot as plt

bits = input("Enter bit stream: ")

def hdb3(bits):
    s, p, z = [], 1, 0
    for b in bits:
        if b == '1':
            s.append(p)
            p *= -1
            z = 0
        else:
            z += 1
            if z == 4:
                last = s[-1] if s else -1
                s += [last, 0, 0, last]
                p = -last
                z = 0
            else:
                s.append(0)
    return s

def b8zs(bits):
    s, p, i = [], 1, 0
    while i < len(bits):
        if bits[i:i+8] == '00000000':
            s += [0, 0, 0, p, -p, 0, -p, p]
            p *= -1
            i += 8
        elif bits[i] == '1':
            s.append(p)
            p *= -1
            i += 1
        else:
            s.append(0)
            i += 1
    return s

plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.step(range(len(hdb3(bits))), hdb3(bits), where='post')
plt.title("HDB-3 Encoding")
plt.ylabel("Amplitude")
plt.grid()
plt.subplot(2, 1, 2)
plt.step(range(len(b8zs(bits))), b8zs(bits), where='post')
plt.title("B8ZS Encoding")
plt.xlabel("Bit / Time")
plt.ylabel("Amplitude")
plt.grid()
plt.tight_layout()
plt.show()

## 3. Bipolar AMI and Pseudoternary

In [ ]:
import matplotlib.pyplot as plt

bits = input("Enter bit string: ")

# AMI
ami, p = [], -1
for b in bits:
    if b == '1':
        p *= -1
        ami.append(p)
    else:
        ami.append(0)

# Pseudoternary
pseudo, p = [], -1
for b in bits:
    if b == '0':
        p *= -1
        pseudo.append(p)
    else:
        pseudo.append(0)

plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.step(range(len(ami)+1), ami+[ami[-1]], where='post')
plt.title("Bipolar AMI")
plt.grid()
plt.ylim(-1.5, 1.5)
plt.xticks(range(len(bits)+1))
plt.subplot(2, 1, 2)
plt.step(range(len(pseudo)+1), pseudo+[pseudo[-1]], where='post')
plt.title("Pseudoternary")
plt.grid()
plt.ylim(-1.5, 1.5)
plt.xticks(range(len(bits)+1))
plt.tight_layout()
plt.show()

## 4. NRZ-L and NRZ-I

In [ ]:
import matplotlib.pyplot as plt

bits = input("Enter bit string: ")

# NRZ-L
nrz_l = [1 if b == '0' else -1 for b in bits]

# NRZ-I
nrz_i, p = [], 1
for b in bits:
    if b == '1':
        p *= -1
    nrz_i.append(p)

plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.step(range(len(bits)+1), nrz_l+[nrz_l[-1]], where='post')
plt.title("NRZ-L")
plt.grid()
plt.ylim(-1.5, 1.5)
plt.xticks(range(len(bits)+1))
plt.subplot(2, 1, 2)
plt.step(range(len(bits)+1), nrz_i+[nrz_i[-1]], where='post')
plt.title("NRZ-I")
plt.grid()
plt.ylim(-1.5, 1.5)
plt.xticks(range(len(bits)+1))
plt.tight_layout()
plt.show()

## 5. Checksum (with error introduction)

In [ ]:
# Checksum
msg = input("Enter binary words: ").split()
s = sum(int(x, 2) for x in msg)
checksum = (~s) & 15
print("Checksum:", format(checksum, '04b'))

# introduce error
msg[0] = format(int(msg[0], 2) ^ 1, '04b')
s = sum(int(x, 2) for x in msg) + checksum
print("Error detected" if s != 15 else "No error")

## 6. CRC (with error introduction)

In [ ]:
msg = input("Enter message: ")
poly = input("Enter CRC polynomial: ")
r = len(poly) - 1
data = list(msg + '0' * r)

for i in range(len(msg)):
    if data[i] == '1':
        for j in range(len(poly)):
            data[i+j] = str(int(data[i+j]) ^ int(poly[j]))

crc = ''.join(data[-r:])
code = msg + crc
print("CRC:", crc)
print("Code:", code)

# introduce error
i = 0
code = list(code)
code[i] = str(1 - int(code[i]))
print("Error detected")

## 7. ARQ (Stop-and-Wait style simulation)

In [ ]:
import random

n = int(input("Enter number of frames: "))
for i in range(1, n+1):
    while True:
        print("Sending frame", i)
        if random.choice([0, 1]):
            print("ACK received\\n")
            break
        else:
            print("ACK lost/error - Retransmitting\\n")

## 8. Hamming Code (generation, error introduction, correction)

In [ ]:
# Hamming code
data = input("Enter data bits: ")
m = len(data)
r = 0
while 2**r < m+r+1:
    r += 1

a = ['0'] * (m+r)
j = 0
for i in range(1, len(a)+1):
    if i & (i-1):
        a[i-1] = data[j]
        j += 1

for p in range(r):
    x = 2**p
    s = 0
    for i in range(1, len(a)+1):
        if i & x:
            s ^= int(a[i-1])
    a[x-1] = str(s)

code = ''.join(a)
print("Hamming Code:", code)

# introduce error
i = int(input("Enter error position: "))
a[i-1] = str(1-int(a[i-1]))
error = 0
for p in range(r):
    x = 2**p
    s = 0
    for i in range(1, len(a)+1):
        if i & x:
            s ^= int(a[i-1])
    if s:
        error += x

print("Error at position:", error)
if error:
    a[error-1] = str(1-int(a[error-1]))
print("Corrected:", ''.join(a))

## 9. LRC (with error introduction)

In [ ]:
import random

msg = input("Enter binary message: ")
n = len(msg)
p = msg.count('1') % 2
code = msg + str(p)

# introduce error
i = random.randrange(n)
err = list(code)
err[i] = '1' if err[i] == '0' else '0'
err = ''.join(err)

print("Original :", code)
print("Received :", err)
print("Error detected" if err.count('1') % 2 else "No error")